# XGBoost and LightGBM Benchmark
Cross-sectional equity return prediction in the Emerging Markets universe. Both models receive the full set of rank-normalised JKP characteristics and are evaluated against the same six-month rebalancing schedule, 25 basis-point transaction cost, and volatility overlay as the Dual Path Portfolio Transformer.

Two portfolio constructions are reported for every model: a long-short quintile portfolio for direct comparison with the dissertation headline Sharpe, and a long-only top-quintile portfolio for comparison with the MSCI EM Gross long-only benchmark. The hyperparameter search optimises validation long-short Sharpe. The long-only metrics are tracked as trial diagnostics and reported at evaluation time for the selected model.

In [ ]:
import json
import time
import pickle
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import xgboost as xgb
import lightgbm as lgb
import optuna
from scipy.stats import spearmanr
import matplotlib
import matplotlib.pyplot as plt

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

In [ ]:

try:
    import torch
    cuda_available = torch.cuda.is_available()
    cuda_device_name = torch.cuda.get_device_name(0) if cuda_available else None
except ImportError:
    cuda_available = False
    cuda_device_name = None

print(f'cuda available (torch):{cuda_available}')
if cuda_available:
    print(f'cuda device:{cuda_device_name}')


def _probe(probe_fn, label):
    try:
        probe_x = np.random.randn(200, 10).astype(np.float32)
        probe_y = np.random.randn(200).astype(np.float32)
        probe_fn(probe_x, probe_y)
        return True
    except Exception as exc:
        print(f'{label}: gpu probe failed ({type(exc).__name__}: {exc})')
        return False


xgb_use_cuda = False
lgb_use_gpu = False
if cuda_available:
    xgb_use_cuda = _probe(
        lambda x, y: xgb.XGBRegressor(
            n_estimators = 5, tree_method = 'hist', device = 'cuda', verbosity = 0,
        ).fit(x, y), 'xgboost',
    )
    lgb_use_gpu = _probe(
        lambda x, y: lgb.LGBMRegressor(
            n_estimators = 5, device = 'gpu', verbose = -1,
        ).fit(x, y), 'lightgbm',
    )

xgb_device_params = {'tree_method': 'hist', 'device': 'cuda'} if xgb_use_cuda else {'tree_method': 'hist'}
lgb_device_params = {'device': 'gpu'} if lgb_use_gpu else {}


In [ ]:
data_path = Path('../data/Global Factor_EM.parquet')
results_dir = Path('../results/benchmark/tree_benchmark')
results_dir.mkdir(parents = True, exist_ok = True)

train_end = pd.Timestamp('2015-12-31')
val_end = pd.Timestamp('2020-12-31')

ret_col = 'ret_exc_lead1m'
rebalance_freq = 3
tc_bps = 25
min_stocks = 30
ret_clip_low = -1.0
ret_clip_high =  1.0

target_vol = 0.10
vol_lookback = 6
max_leverage_ls = 3.0
max_leverage_lo = 3.0

n_trials_xgb = 50
n_trials_lgb = 50
optuna_seed = 42
n_hpo_months = 36

In [ ]:
schema = pq.read_schema(data_path)

non_feature = {
    'id', 'gvkey', 'isin', 'cusip', 'permno', 'permco',
    'eom', 'excntry', 'sic', 'naics', 'source_crsp', ret_col,
}
feature_cols = [
    c for c in schema.names
    if c not in non_feature
    and 'float' in str(schema.field(c).type).lower()
]

needed = list(dict.fromkeys(
    [c for c in ['id', 'eom', 'excntry', ret_col] + feature_cols
     if c in schema.names]
))

df = pd.read_parquet(data_path, columns = needed)
df['eom'] = pd.to_datetime(df['eom'])

for col in feature_cols:
    if col in df.columns and df[col].dtype == np.float64:
        df[col] = df[col].astype(np.float32)

df[ret_col] = df[ret_col].clip(lower = ret_clip_low, upper = ret_clip_high)

print(f'loaded:{df.shape[0]:,} rows, {len(feature_cols)} characteristic columns')
print(f'date range:{df["eom"].min().date()} to {df["eom"].max().date()}')
print(f'countries{df["excntry"].nunique()}')

In [ ]:
sorted_eoms = sorted(df['eom'].unique())
all_months = {}
n_feat = len(feature_cols)

for eom in sorted_eoms:
    month = df[df['eom'] == eom].copy()
    month = month[month[ret_col].notna()]
    if len(month) < min_stocks:
        continue

    ids = month['id'].values
    r   = month[ret_col].values.astype(np.float64)

    x = np.full((len(month), n_feat), np.nan, dtype = np.float32)
    for j, col in enumerate(feature_cols):
        if col not in month.columns:
            continue
        vals = month[col].astype(np.float64).to_numpy()
        valid = np.isfinite(vals)
        if valid.sum() > 1:
            ranks = pd.Series(vals[valid]).rank(pct = True).to_numpy(dtype = np.float32)
            x[valid, j] = ranks - 0.5

    all_months[eom] = {'ids': ids, 'r': r, 'x': x}

sorted_dates = sorted(all_months.keys())
print(f'processed: {len(sorted_dates)} months')
print(f'avg firms/month: {np.mean([len(m["ids"]) for m in all_months.values()]):.0f}')

In [ ]:
train_dates = [d for d in sorted_dates if d <= train_end]
val_dates = [d for d in sorted_dates if train_end < d <= val_end]
test_dates = [d for d in sorted_dates if d > val_end]

print(f'train:{len(train_dates)} months')
print(f'val:{len(val_dates)} months')
print(f'test:{len(test_dates)} months')

x_train = np.vstack([all_months[d]['x'] for d in train_dates])
y_train = np.concatenate([all_months[d]['r'] for d in train_dates])
print(f'x_train: {x_train.shape}')

hpo_dates = train_dates[-n_hpo_months:]
x_hpo = np.vstack([all_months[d]['x'] for d in hpo_dates])
y_hpo = np.concatenate([all_months[d]['r'] for d in hpo_dates])
print(f'x_hpo:{x_hpo.shape}')

In [ ]:
def portfolio_metrics(rets):
    rets = np.array(rets, dtype = np.float64)
    if len(rets) == 0:
        return {}
    tw = float((1.0 + rets).prod())
    ann_ret = -1.0 if tw <= 0 else float(tw ** (12.0 / len(rets)) - 1.0)
    ann_vol = float(rets.std() * np.sqrt(12.0))
    sharpe = ann_ret / max(ann_vol, 1e-8)
    se = float(np.sqrt((1.0 + 0.5 * sharpe ** 2) / len(rets)))
    cw = np.cumprod(1.0 + rets)
    pk = np.maximum.accumulate(cw)
    max_dd = float(((pk - cw) / pk).max()) if len(cw) > 0 else 0.0
    return {
        'ann_ret': ann_ret, 'ann_vol': ann_vol, 'sharpe': sharpe,
        'se_sharpe': se, 'max_dd': max_dd, 'n_obs': len(rets),
    }


def apply_vol_target(monthly_rets, rebalance_indices, target_vol, vol_lookback, max_leverage):
    scaled = np.array(monthly_rets, dtype = np.float64)
    n = len(monthly_rets)
    n_rb = len(rebalance_indices)
    period_rets = []
    for i in range(1, n_rb):
        window = np.array(monthly_rets[rebalance_indices[i - 1]:rebalance_indices[i]])
        period_rets.append(float(np.prod(1.0 + window) - 1.0))
    for i in range(n_rb):
        if i < vol_lookback:
            continue
        trailing = np.array(period_rets[max(0, i - vol_lookback):i])
        if len(trailing) < 2:
            continue
        sigma_ann = float(trailing.std() * np.sqrt(12.0 / rebalance_freq))
        lev = float(np.clip(target_vol / max(sigma_ann, 1e-8), 1.0 / max_leverage, max_leverage))
        next_rb = rebalance_indices[i + 1] if i + 1 < n_rb else n
        scaled[rebalance_indices[i]:next_rb] = (
            np.array(monthly_rets[rebalance_indices[i]:next_rb]) * lev
        )
    return scaled


def predict_test(model, month_dates):
    rows = []
    for eom in month_dates:
        if eom not in all_months:
            continue
        m = all_months[eom]
        pred = model.predict(m['x'])
        for k in range(len(m['ids'])):
            rows.append({
                'eom': eom, 'id': m['ids'][k],
                'prediction': float(pred[k]), 'realised_return': float(m['r'][k]),
            })
    return pd.DataFrame(rows)


def run_quintile_simulation(model, month_dates):
    """
    Quintile simulation producing both long-short and long-only portfolios.
    Long-short: top quintile equal-weighted long, bottom quintile equal-weighted short.
    Long-only: top quintile equal-weighted, no short leg.
    Turnover is computed per leg using firm identifiers.
    The long-only turnover counts only the long leg; the long-short turnover
    counts both legs.
    """
    rset = set(month_dates[::rebalance_freq])

    ls_rets, ls_dates, ls_rb_indices = [], [], []
    lo_rets, lo_dates, lo_rb_indices = [], [], []

    li_ids = []
    si_ids = []
    prev_li = set()
    prev_si = set()

    ls_holdings, lo_holdings = [], []
    rb_counter = -1

    for eom in month_dates:
        if eom not in all_months:
            continue
        m = all_months[eom]
        ids = m['ids']
        r = m['r']
        x = m['x']

        ls_tcv = 0.0
        lo_tcv = 0.0

        if eom in rset:
            ls_rb_indices.append(len(ls_rets))
            lo_rb_indices.append(len(lo_rets))
            rb_counter += 1

            pred = model.predict(x)
            valid = np.isfinite(pred)
            if valid.sum() < 10:
                continue

            vi = ids[valid]
            vp = pred[valid]
            nq = max(1, int(len(vi) * 0.20))
            so = np.argsort(vp)
            li_ids = vi[so[::-1][:nq]].tolist()
            si_ids = vi[so[:nq]].tolist()
            li_set = set(li_ids)
            si_set = set(si_ids)

            ls_to = (
                len(li_set - prev_li) + len(prev_li - li_set)
                + len(si_set - prev_si) + len(prev_si - si_set)
            ) / max(nq, 1)
            ls_tcv = ls_to * tc_bps / 10000.0

            lo_to = (
                len(li_set - prev_li) + len(prev_li - li_set)
            ) / max(nq, 1)
            lo_tcv = lo_to * tc_bps / 10000.0

            prev_li = li_set
            prev_si = si_set

            wt_long = 1.0 / max(len(li_ids), 1)
            wt_short = -1.0 / max(len(si_ids), 1)

            for fid in li_ids:
                ls_holdings.append({
                    'rebalance_index': rb_counter, 'eom': eom, 'leg': 'long',
                    'id': fid, 'weight': wt_long,
                })
                lo_holdings.append({
                    'rebalance_index': rb_counter, 'eom': eom, 'leg': 'long',
                    'id': fid, 'weight': wt_long,
                })
            for fid in si_ids:
                ls_holdings.append({
                    'rebalance_index': rb_counter, 'eom': eom, 'leg': 'short',
                    'id': fid, 'weight': wt_short,
                })

        if not li_ids:
            continue

        li_mask = np.isin(ids, li_ids)
        si_mask = np.isin(ids, si_ids)
        lr = r[li_mask]
        sr = r[si_mask]
        lr_mean = float(lr.mean()) if len(lr) > 0 else 0.0
        sr_mean = float(sr.mean()) if len(sr) > 0 else 0.0

        ls_rets.append(lr_mean - sr_mean - ls_tcv)
        ls_dates.append(eom)

        lo_rets.append(lr_mean - lo_tcv)
        lo_dates.append(eom)

    return {
        'long_short': {
            'returns': np.array(ls_rets),
            'rb_indices': ls_rb_indices,
            'holdings_df': pd.DataFrame(ls_holdings),
            'returns_df': pd.DataFrame({'eom': ls_dates, 'return_raw': ls_rets}),
        },
        'long_only': {
            'returns': np.array(lo_rets),
            'rb_indices': lo_rb_indices,
            'holdings_df': pd.DataFrame(lo_holdings),
            'returns_df': pd.DataFrame({'eom': lo_dates, 'return_raw': lo_rets}),
        },
    }

### Hyperparameter search

In [ ]:
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log = True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 5.0, log = True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 5.0, log = True),
        'random_state': optuna_seed,
        'n_jobs': -1,
        'verbosity': 0,
        **xgb_device_params,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(x_hpo, y_hpo)

    sim = run_quintile_simulation(model, val_dates)
    ls = sim['long_short']
    lo = sim['long_only']

    if len(ls['returns']) == 0:
        return -999.0

    ls_scaled = apply_vol_target(ls['returns'], ls['rb_indices'], target_vol, vol_lookback, max_leverage_ls)
    lo_scaled = apply_vol_target(lo['returns'], lo['rb_indices'], target_vol, vol_lookback, max_leverage_lo)

    ls_sharpe = portfolio_metrics(ls_scaled).get('sharpe', -999.0)
    lo_sharpe = portfolio_metrics(lo_scaled).get('sharpe', -999.0)

    trial.set_user_attr('val_sharpe_long_only', float(lo_sharpe))
    return ls_sharpe


xgb_study = optuna.create_study(
    direction = 'maximize',
    sampler = optuna.samplers.TPESampler(seed = optuna_seed)
)
t0 = time.time()
xgb_study.optimize(xgb_objective, n_trials = n_trials_xgb, show_progress_bar = True)
xgb_hpo_time = time.time() - t0
xgb_best = xgb_study.best_params
print(f'XGBoost best val ls sharpe: {xgb_study.best_value:.4f}')
print(f'XGBoost best params: {xgb_best}')
print(f'XGBoost hpo time:{xgb_hpo_time:.1f}s ({xgb_hpo_time / 60:.2f} min)')

def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log = True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 5.0, log = True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 5.0, log = True),
        'random_state': optuna_seed,
        'n_jobs': -1,
        'verbose': -1,
        **lgb_device_params,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(x_hpo, y_hpo)

    sim = run_quintile_simulation(model, val_dates)
    ls = sim['long_short']
    lo = sim['long_only']

    if len(ls['returns']) == 0:
        return -999.0

    ls_scaled = apply_vol_target(ls['returns'], ls['rb_indices'], target_vol, vol_lookback, max_leverage_ls)
    lo_scaled = apply_vol_target(lo['returns'], lo['rb_indices'], target_vol, vol_lookback, max_leverage_lo)

    ls_sharpe = portfolio_metrics(ls_scaled).get('sharpe', -999.0)
    lo_sharpe = portfolio_metrics(lo_scaled).get('sharpe', -999.0)

    trial.set_user_attr('val_sharpe_long_only', float(lo_sharpe))
    return ls_sharpe


lgb_study = optuna.create_study(
    direction = 'maximize',
    sampler = optuna.samplers.TPESampler(seed = optuna_seed)
)
t0 = time.time()
lgb_study.optimize(lgb_objective, n_trials = n_trials_lgb, show_progress_bar = True)
lgb_hpo_time = time.time() - t0
lgb_best = lgb_study.best_params
print(f'LightGBM best val ls sharpe:{lgb_study.best_value:.4f}')
print(f'LightGBM best params:{lgb_best}')
print(f'LightGBM hpo time:{lgb_hpo_time:.1f} s ({lgb_hpo_time / 60:.2f} min)')

In [ ]:
xgb_trials_df = xgb_study.trials_dataframe()
xgb_trials_df.to_csv(results_dir / 'em_xgb_optuna_trials.csv', index = False)
with open(results_dir / 'em_xgb_optuna_study.pkl', 'wb') as fh:
    pickle.dump(xgb_study, fh)

lgb_trials_df = lgb_study.trials_dataframe()
lgb_trials_df.to_csv(results_dir / 'em_lgb_optuna_trials.csv', index = False)
with open(results_dir / 'em_lgb_optuna_study.pkl', 'wb') as fh:
    pickle.dump(lgb_study, fh)

print(f'xgboost trials saved:{len(xgb_trials_df)} rows')
print(f'lightgbm trials saved: {len(lgb_trials_df)} rows')

In [ ]:
xgb_model = xgb.XGBRegressor(
    **xgb_best, random_state = optuna_seed, n_jobs = -1, verbosity = 0,
    **xgb_device_params,
)
t0 = time.time()
xgb_model.fit(x_train, y_train)
xgb_train_time = time.time() - t0
print(f'XGBoost final model: trained in {xgb_train_time:.1f} s')

lgb_model = lgb.LGBMRegressor(
    **lgb_best, random_state = optuna_seed, n_jobs = -1, verbose = -1,
    **lgb_device_params,
)
t0 = time.time()
lgb_model.fit(x_train, y_train)
lgb_train_time = time.time() - t0
print(f'LightGBM final model: trained in {lgb_train_time:.1f} s')

xgb_model.save_model(str(results_dir / 'em_xgb_model.json'))
lgb_model.booster_.save_model(str(results_dir / 'em_lgb_model.txt'))
print('models saved in native formats')

In [ ]:
def rank_correlation_oos(model, month_dates):
    corrs = []
    for eom in month_dates:
        if eom not in all_months:
            continue
        m = all_months[eom]
        pred = model.predict(m['x'])
        valid = np.isfinite(pred) & np.isfinite(m['r'])
        if valid.sum() < 10:
            continue
        c, _ = spearmanr(pred[valid], m['r'][valid])
        if not np.isnan(c):
            corrs.append(float(c))
    return float(np.mean(corrs)) if corrs else 0.0


xgb_rc_val = rank_correlation_oos(xgb_model, val_dates)
xgb_rc_test = rank_correlation_oos(xgb_model, test_dates)
lgb_rc_val = rank_correlation_oos(lgb_model, val_dates)
lgb_rc_test = rank_correlation_oos(lgb_model, test_dates)

print(f'XGBoost rank corr: val = {xgb_rc_val:.4f}, test = {xgb_rc_test:.4f}')
print(f'LightGBM rank corr: val = {lgb_rc_val:.4f}, test = {lgb_rc_test:.4f}')

In [ ]:
# test set simulation: both long-short and long-only for each model

def evaluate_and_save(model, name):
    sim = run_quintile_simulation(model, test_dates)
    ls = sim['long_short']
    lo = sim['long_only']

    ls_scaled = apply_vol_target(ls['returns'], ls['rb_indices'], target_vol, vol_lookback, max_leverage_ls)
    lo_scaled = apply_vol_target(lo['returns'], lo['rb_indices'], target_vol, vol_lookback, max_leverage_lo)

    ls['returns_df']['return_scaled'] = ls_scaled
    lo['returns_df']['return_scaled'] = lo_scaled

    m_ls_raw = portfolio_metrics(ls['returns'])
    m_ls_scaled = portfolio_metrics(ls_scaled)
    m_lo_raw = portfolio_metrics(lo['returns'])
    m_lo_scaled = portfolio_metrics(lo_scaled)

    ls['returns_df'].to_parquet(results_dir / f'em_{name}_returns_long_short.parquet', index = False)
    lo['returns_df'].to_parquet(results_dir / f'em_{name}_returns_long_only.parquet', index = False)
    ls['holdings_df'].to_parquet(results_dir / f'em_{name}_holdings_long_short.parquet', index = False)
    lo['holdings_df'].to_parquet(results_dir / f'em_{name}_holdings_long_only.parquet', index = False)

    predict_test(model, test_dates).to_parquet(
        results_dir / f'em_{name}_test_predictions.parquet', index = False,
    )

    return {
        'returns_ls_raw': ls['returns'],
        'returns_ls_scaled': ls_scaled,
        'returns_lo_raw': lo['returns'],
        'returns_lo_scaled': lo_scaled,
        'metrics':{
            'long_short_raw': m_ls_raw,
            'long_short_scaled': m_ls_scaled,
            'long_only_raw': m_lo_raw,
            'long_only_scaled':  m_lo_scaled,
        },
    }


xgb_eval = evaluate_and_save(xgb_model, 'xgb')
lgb_eval = evaluate_and_save(lgb_model, 'lgb')

for name, ev in [('XGBoost', xgb_eval), ('LightGBM', lgb_eval)]:
    mls = ev['metrics']['long_short_scaled']
    mlo = ev['metrics']['long_only_scaled']
    print(f'{name} long-short (vol):sharpe = {mls["sharpe"]:.4f}, ann_ret = {mls["ann_ret"] * 100:.2f}%, ann_vol = {mls["ann_vol"] * 100:.2f}%')
    print(f'{name} long-only (vol):sharpe = {mlo["sharpe"]:.4f}, ann_ret = {mlo["ann_ret"] * 100:.2f}%, ann_vol = {mlo["ann_vol"] * 100:.2f}%')

In [ ]:
xgb_imp = pd.DataFrame({
    'feature': feature_cols, 'importance': xgb_model.feature_importances_,
}).sort_values('importance', ascending = False)
lgb_imp = pd.DataFrame({
    'feature': feature_cols, 'importance': lgb_model.feature_importances_,
}).sort_values('importance', ascending = False)

xgb_imp.to_csv(results_dir / 'em_xgb_feature_importance.csv', index = False)
lgb_imp.to_csv(results_dir / 'em_lgb_feature_importance.csv', index = False)

print('top 10 xgboost features')
print(xgb_imp.head(10).to_string(index = False))
print('top 10 lightgbm features')
print(lgb_imp.head(10).to_string(index = False))

In [ ]:
summary = {
    'n_features': len(feature_cols),
    'feature_cols': feature_cols,
    'split': {
        'train':{'start': str(train_dates[0].date()), 'end': str(train_dates[-1].date()),
                  'n_months': len(train_dates), 'n_obs': int(x_train.shape[0])},
        'val': {'start': str(val_dates[0].date()), 'end': str(val_dates[-1].date()),
                  'n_months': len(val_dates)},
        'test':{'start': str(test_dates[0].date()), 'end': str(test_dates[-1].date()),
                  'n_months': len(test_dates)},
        'hpo':{'start': str(hpo_dates[0].date()), 'end': str(hpo_dates[-1].date()),
                  'n_months': len(hpo_dates), 'n_obs': int(x_hpo.shape[0])},
    },
    'config': {
        'rebalance_freq': rebalance_freq, 'tc_bps': tc_bps, 'min_stocks': min_stocks,
        'ret_clip': [ret_clip_low, ret_clip_high],
        'target_vol': target_vol, 'vol_lookback': vol_lookback,
        'max_leverage_ls': max_leverage_ls, 'max_leverage_lo': max_leverage_lo,
        'optuna_seed': optuna_seed,
        'n_trials_xgb': n_trials_xgb, 'n_trials_lgb': n_trials_lgb,
    },
    'xgboost': {
        'best_params': xgb_best,
        'best_val_long_short_sharpe': float(xgb_study.best_value),
        'best_trial_number': int(xgb_study.best_trial.number),
        'n_trials_completed': sum(1 for t in xgb_study.trials if t.state.name == 'COMPLETE'),
        'hpo_time_seconds': float(xgb_hpo_time),
        'final_training_time_seconds': float(xgb_train_time),
        'rc_val': float(xgb_rc_val), 'rc_test': float(xgb_rc_test),
        'portfolio_metrics': xgb_eval['metrics'],
    },
    'lightgbm': {
        'best_params': lgb_best,
        'best_val_long_short_sharpe': float(lgb_study.best_value),
        'best_trial_number': int(lgb_study.best_trial.number),
        'n_trials_completed': sum(1 for t in lgb_study.trials if t.state.name == 'COMPLETE'),
        'hpo_time_seconds': float(lgb_hpo_time),
        'final_training_time_seconds': float(lgb_train_time),
        'rc_val': float(lgb_rc_val), 'rc_test': float(lgb_rc_test),
        'portfolio_metrics': lgb_eval['metrics'],
    },
}

with open(results_dir / 'em_tree_summary.json', 'w') as fh:
    json.dump(summary, fh, indent = 2, default = float)

print(f'summary saved to {results_dir / "em_tree_summary.json"}')

print('nTree Benchmark: EM Universe (test set, vol-targeted)')
print(f'{"model":<12} {"portfolio":<14} {"rc_test":>8} {"sharpe":>8} {"se":>8} {"ann_ret":>9} {"ann_vol":>9} {"max_dd":>8}')
for name, ev, rc in [('xgboost', xgb_eval, xgb_rc_test), ('lightgbm', lgb_eval, lgb_rc_test)]:
    for label, key in [('long_short', 'long_short_scaled'), ('long_only', 'long_only_scaled')]:
        m = ev['metrics'][key]
        print(
            f'{name:<12} {label:<14} {rc:8.4f} {m["sharpe"]:8.4f} {m["se_sharpe"]:8.4f}'
            f'{m["ann_ret"] * 100:8.2f}% {m["ann_vol"] * 100:8.2f}% {m["max_dd"] * 100:7.2f}%'
        )

In [ ]:
matplotlib.rcParams['font.size'] = 12

fig, axes = plt.subplots(2, 3, figsize = (18, 10))
fig.suptitle('XGBoost and LightGBM Benchmark', fontsize = 14)

# cumulative wealth, long-short
axes[0, 0].plot(np.cumprod(1.0 + xgb_eval['returns_ls_scaled']), label = 'XGBoost', color = 'steelblue')
axes[0, 0].plot(np.cumprod(1.0 + lgb_eval['returns_ls_scaled']), label = 'LightGBM', color = 'darkorange')
axes[0, 0].set_title('Long-Short Cumulative Wealth (vol-targeted)')
axes[0, 0].legend()
axes[0, 0].grid(alpha = 0.3)

# cumulative wealth, long-only
axes[0, 1].plot(np.cumprod(1.0 + xgb_eval['returns_lo_scaled']), label = 'XGBoost', color = 'steelblue')
axes[0, 1].plot(np.cumprod(1.0 + lgb_eval['returns_lo_scaled']), label = 'LightGBM', color = 'darkorange')
axes[0, 1].set_title('Long-Only Cumulative Wealth (vol-targeted)')
axes[0, 1].legend()
axes[0, 1].grid(alpha = 0.3)

# sharpe bars side by side
names = ['XGBoost', 'LightGBM']
ls_sharpes = [xgb_eval['metrics']['long_short_scaled']['sharpe'], lgb_eval['metrics']['long_short_scaled']['sharpe']]
lo_sharpes = [xgb_eval['metrics']['long_only_scaled']['sharpe'],  lgb_eval['metrics']['long_only_scaled']['sharpe']]
x_pos = np.arange(len(names))
axes[0, 2].bar(x_pos - 0.2, ls_sharpes, width = 0.4, label = 'long-short', color = 'steelblue')
axes[0, 2].bar(x_pos + 0.2, lo_sharpes, width = 0.4, label = 'long-only',  color = 'darkorange')
axes[0, 2].set_xticks(x_pos)
axes[0, 2].set_xticklabels(names)
axes[0, 2].set_title('Sharpe Ratios (vol-targeted)')
axes[0, 2].legend()
axes[0, 2].grid(axis = 'y', alpha = 0.3)

# optuna search histories
xgb_vals = [t.value for t in xgb_study.trials if t.value is not None]
axes[1, 0].plot(np.maximum.accumulate(xgb_vals), color = 'steelblue')
axes[1, 0].scatter(range(len(xgb_vals)), xgb_vals, alpha = 0.3, s = 15, color = 'steelblue')
axes[1, 0].set_xlabel('trial'); axes[1, 0].set_ylabel('val ls sharpe')
axes[1, 0].set_title('XGBoost Optuna Search'); axes[1, 0].grid(alpha = 0.3)

lgb_vals = [t.value for t in lgb_study.trials if t.value is not None]
axes[1, 1].plot(np.maximum.accumulate(lgb_vals), color = 'darkorange')
axes[1, 1].scatter(range(len(lgb_vals)), lgb_vals, alpha = 0.3, s = 15, color = 'darkorange')
axes[1, 1].set_xlabel('trial'); axes[1, 1].set_ylabel('val ls sharpe')
axes[1, 1].set_title('LightGBM Optuna Search'); axes[1, 1].grid(alpha = 0.3)

# xgboost feature importance
top_xgb_imp = xgb_imp.head(15)
axes[1, 2].barh(range(len(top_xgb_imp)), top_xgb_imp['importance'][::-1], color = 'steelblue')
axes[1, 2].set_yticks(range(len(top_xgb_imp)))
axes[1, 2].set_yticklabels(top_xgb_imp['feature'][::-1], fontsize = 8)
axes[1, 2].set_title('Top 15 XGBoost Features')
axes[1, 2].grid(axis = 'x', alpha = 0.3)

plt.tight_layout()
plt.savefig(results_dir / 'em_tree_benchmark.png', dpi = 150, bbox_inches = 'tight')
plt.show()
print(f'plot saved: {results_dir / "em_tree_benchmark.png"}')